# P0：开发期探索性分析

分析窗口为 2026 年 4–7 月，以 **prediction_time 的 UTC 半开区间**界定。8 月作为最终测试月，开发期不查看其目标分布。这里的分析单位是符合 RQ2 条件的同机相邻航段衔接；不是所有 BTS 航班。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

if "ROOT" not in globals():
    ROOT = Path.cwd()
    while not (ROOT / "scripts/build_features.py").exists():
        if ROOT.parent == ROOT:
            raise RuntimeError("请在 Airline-delay 仓库中运行 Notebook")
        ROOT = ROOT.parent
if "RUN_DIR" not in globals():
    candidates = [p for p in (ROOT / "experiments/runs").glob("R*")
                  if (p / "metrics.csv").exists() and (p / "eda_summary.json").exists()
                  and (p / "rq1_summary.json").exists()]
    if not candidates:
        raise RuntimeError("请先运行 P0 开发分析，或手动设置 RUN_DIR")
    RUN_DIR = max(candidates, key=lambda p: int(p.name[1:]))
TABLES = RUN_DIR / "tables"
FIGURES = RUN_DIR / "figures"
def table(name):
    return pd.read_csv(TABLES / name)
def figure(name):
    display(Image(filename=str(FIGURES / name)))
def receipt(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
display(Markdown(f"**归档运行：{RUN_DIR.name}**。本 Notebook 读取该轮表格与图，不重新拟合模型。"))

## 1. 样本定义与排除

预测时点为上游实际到达，标签时点为下游实际出发。保留有限且非负的下游出发延误，要求 prediction_time < label_time；同刻衔接因为下游已经出发而排除。上游准点、非隔夜与隔夜记录均可进入 RQ2，隔夜误差另外评价。

In [ ]:
summary = receipt("eda_summary.json")
display(pd.DataFrame([{"指标": k, "值": v} for k, v in summary.items() if k not in ["feature_audit", "heatmap_note"]]))
audit = summary.get("feature_audit", {}).get("counts", {})
display(pd.DataFrame(list(audit.items()), columns=["顺序样本流", "数量"]))

## 2. 目标分布与月份漂移

目标使用下游**非负出发延误**，缺失标签不填零。直方分组覆盖全部右尾，最后一档保留极端延误；分位数表同时给出样本量、均值、中位数和 p90/p95/p99，避免均值掩盖偏态。月份只用于漂移和分层，不进入 G0 默认模型。

In [ ]:
display(table("eda_monthly_target.csv"))
display(table("eda_target_bins.csv"))
figure("eda_target_distribution.png")
figure("eda_monthly_target.png")

## 3. 上游延误 × 计划周转

计划周转是计划到达至计划出发的间隔，不等于已知作业缓冲。色图分开呈现支持量与平均 \(\Delta = D_{down}^{dep} - D_{up}^{arr}\)（分钟）。空样本均值格保留缺失；计数格明确为零。负值表示样本衔接中的净延误减少，不自动代表机场造成了减少。

In [ ]:
figure("eda_turnaround_count.png")
figure("eda_turnaround_mean_delta.png")
display(table("eda_turnaround_heatmap.csv").head(12))

## 4. 机场、航线、航司支持与字段缺失

图按衔接量选出最多的 20 个机场，完整支持表保留所有机场。BTS 航班量与有效同机衔接量是不同分母。稀疏机场或类别可能导致估计不稳定；编码与填补只在每折训练段拟合。此处只描述组成，不作因果判断。

In [ ]:
figure("eda_airport_support.png")
for name in ["airport", "route", "airline"]:
    display(Markdown(f"**{name} 支持（前20行）**"))
    display(table(f"eda_support_{name}.csv").head(20))
display(table("eda_feature_missing.csv"))

## 5. P0 的解释边界

这轮用于建立统一样本、时间规则和初始参照。天气、飞机家族、近期机场运行等信息尚未加入；后续每次只改变一个研究因素，继续使用相同评估 ID。RQ1 主样本与本页 RQ2 样本不同，请参阅下一本 Notebook。